# Regression Evaluation Metrics

Evaluating model performance is a crucial step in the machine learning workflow. For regression problems (where the target variable is continuous), we use specific regression metrics to measure how close our model's predictions are to the actual values.

---

## Why Do We Need Multiple Metrics?
No single metric is perfect. A model might perform well under one metric but show weaknesses under another. Different metrics help us understand:
- The **average magnitude** of error (MAE, RMSE)
- The **sensitivity to large errors** or outliers (MSE, RMSE)
- The **relative performance** against a simple baseline (R² Score)
- The **impact of feature complexity** (Adjusted R² Score)

---


## 1. Error Magnitude Metrics

Here is a summary of the three most common error magnitude metrics:

| Metric | Formula | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Mean Absolute Error (MAE)** | $$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$ | • Interpretable in the same units as target variable (e.g., LPA).<br>• Robust to outliers (does not square the errors). | • The absolute value function is **not differentiable at zero**.<br>• Makes optimization via gradient descent mathematically difficult. |
| **Mean Squared Error (MSE)** | $$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$ | • Differentiable everywhere.<br>• Excellent loss function for optimization algorithms. | • Highly **sensitive to outliers** (squaring penalizes large errors heavily).<br>• Units are squared (e.g., $\text{LPA}^2$), making direct interpretation difficult. |
| **Root Mean Squared Error (RMSE)** | $$\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$ | • Returns error to the **original units** of the target variable.<br>• Easier to interpret than MSE while preserving differentiability for loss. | • Still relatively sensitive to outliers compared to MAE. |

> 💡 **Key Insight:** Choose **MAE** when you want a robust average error that isn't dominated by extreme outliers. Choose **RMSE/MSE** when you want to heavily penalize large errors (e.g., in critical safety or finance models where large errors are catastrophic).

## 2. Goodness-of-Fit Metrics

While MAE and RMSE tell us the absolute error, they don't tell us how good the model is relative to the variance of the data. For this, we use $R^2$ and Adjusted $R^2$.

### 2.1 $R^2$ Score (Coefficient of Determination)
The $R^2$ score compares our regression model to a **baseline model**—which is simply predicting the **mean** of the target variable ($\bar{y}$) for all observations.

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

Where:
- **$SS_{res}$ (Sum of Squared Residuals):** The sum of squared errors of our model. $SS_{res} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$
- **$SS_{tot}$ (Total Sum of Squares):** The variance of the target variable around its mean. $SS_{tot} = \sum_{i=1}^{n} (y_i - \bar{y})^2$

#### Interpretation of $R^2$:
- **$R^2 = 1$**: Perfect model. Predictions match actual values exactly.
- **$R^2 = 0$**: The model performs exactly as well as predicting the average (mean).
- **$R^2 < 0$**: The model is worse than predicting the average (usually happens when the model is highly overfitted or inappropriate for the data).

---

### 2.2 Adjusted $R^2$ Score
**The Flaw of $R^2$:** If you add a new feature to your model—even if it is completely random noise (like a student's favorite color)—the $R^2$ score will **always increase or stay the same**. It will never decrease. This can mislead you into thinking your model is improving when it is actually overfitting.

**The Solution:** The Adjusted $R^2$ score penalizes the model for adding features that do not contribute to predicting the target.

$$R^2_{adj} = 1 - \frac{(1 - R^2)(n - 1)}{n - k - 1}$$

Where:
- **$n$**: Number of data points (observations).
- **$k$**: Number of independent features (predictors).

#### Why it's useful:
- If a new feature **improves the model significantly**, $R^2$ increases enough to outweigh the penalty, and $R^2_{adj}$ **increases**.
- If a new feature is **useless/irrelevant**, $R^2$ increases very little, but $k$ increases, causing $R^2_{adj}$ to **decrease**.

---


## 3. Code Implementation using Scikit-Learn

Let's write a practical demonstration of these metrics using python. We will build a synthetic dataset for student packages, train two linear regression models (one with relevant features only, and one with irrelevant noise), and compare the metrics.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# 1. Generate Synthetic Placement Dataset
np.random.seed(42)
num_samples = 150

# Features: CGPA (5.0 to 10.0), IQ (80 to 140)
cgpa = np.random.uniform(5.0, 10.0, num_samples)
iq = np.random.uniform(80, 140, num_samples)

# Target: Package (LPA) = 1.2 * CGPA + 0.05 * IQ - 4 + noise
noise = np.random.normal(0, 0.5, num_samples)
package = 1.2 * cgpa + 0.05 * iq - 4 + noise

# Irrelevant Features (pure noise)
random_feature_1 = np.random.uniform(0, 100, num_samples)  # Random noise
random_feature_2 = np.random.uniform(10, 50, num_samples)   # More random noise

# Create DataFrame
df = pd.DataFrame({
    'CGPA': cgpa,
    'IQ': iq,
    'Random_Feature_1': random_feature_1,
    'Random_Feature_2': random_feature_2,
    'Package_LPA': package
})

print("Dataset generated successfully!")
print(df.head())


In [ ]:
# Define a helper function to calculate Adjusted R2
def adjusted_r2_score(r2, n, k):
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


## 3.1 Model Comparison: R² vs. Adjusted R²

We will train two models:
1. **Model 1 (Good Model):** Trained only on relevant features (`CGPA` and `IQ`).
2. **Model 2 (Useless Features Model):** Trained on relevant features AND two completely random noise features (`Random_Feature_1`, `Random_Feature_2`).

In [ ]:
# Model 1: Relevant features only
X1 = df[['CGPA', 'IQ']]
y = df['Package_LPA']

# Split data
X1_train, X1_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, random_state=42)

# Train model
model1 = LinearRegression()
model1.fit(X1_train, y_train)

# Predict
y_pred1 = model1.predict(X1_test)

# Calculate metrics
mae1 = mean_absolute_error(y_test, y_pred1)
mse1 = mean_squared_error(y_test, y_pred1)
rmse1 = np.sqrt(mse1)
r2_1 = r2_score(y_test, y_pred1)

# Adjusted R2 (n = test set size, k = number of features)
n1 = X1_test.shape[0]
k1 = X1_test.shape[1]
adj_r2_1 = adjusted_r2_score(r2_1, n1, k1)

print("="*50)
print("MODEL 1 METRICS (CGPA + IQ)")
print("="*50)
print(f"MAE:         {mae1:.4f} LPA")
print(f"MSE:         {mse1:.4f}")
print(f"RMSE:        {rmse1:.4f} LPA")
print(f"R2 Score:    {r2_1:.6f}")
print(f"Adjusted R2: {adj_r2_1:.6f}")
print("="*50)


In [ ]:
# Model 2: Relevant features + Irrelevant random noise features
X2 = df[['CGPA', 'IQ', 'Random_Feature_1', 'Random_Feature_2']]

# Split data (using the same random state for exact comparison)
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.2, random_state=42)

# Train model
model2 = LinearRegression()
model2.fit(X2_train, y_train)

# Predict
y_pred2 = model2.predict(X2_test)

# Calculate metrics
mae2 = mean_absolute_error(y_test, y_pred2)
mse2 = mean_squared_error(y_test, y_pred2)
rmse2 = np.sqrt(mse2)
r2_2 = r2_score(y_test, y_pred2)

# Adjusted R2 (n = test set size, k = number of features)
n2 = X2_test.shape[0]
k2 = X2_test.shape[1]
adj_r2_2 = adjusted_r2_score(r2_2, n2, k2)

print("="*50)
print("MODEL 2 METRICS (CGPA + IQ + 2 Random Noise Features)")
print("="*50)
print(f"MAE:         {mae2:.4f} LPA")
print(f"MSE:         {mse2:.4f}")
print(f"RMSE:        {rmse2:.4f} LPA")
print(f"R2 Score:    {r2_2:.6f}")
print(f"Adjusted R2: {adj_r2_2:.6f}")
print("="*50)


In [ ]:
# Compare metrics in a DataFrame
comparison_df = pd.DataFrame({
    'Metric': ['MAE (LPA)', 'MSE', 'RMSE (LPA)', 'R2 Score', 'Adjusted R2 Score'],
    'Model 1 (CGPA + IQ)': [mae1, mse1, rmse1, r2_1, adj_r2_1],
    'Model 2 (+ Useless Features)': [mae2, mse2, rmse2, r2_2, adj_r2_2]
})

print("METRIC COMPARISON TABLE:")
print(comparison_df.to_string(index=False))

# Visualize the R2 vs Adjusted R2 behavior
metrics = ['R2 Score', 'Adjusted R2 Score']
model1_scores = [r2_1, adj_r2_1]
model2_scores = [r2_2, adj_r2_2]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, model1_scores, width, label='Model 1 (CGPA+IQ)', color='#3498db')
rects2 = ax.bar(x + width/2, model2_scores, width, label='Model 2 (+ Noise)', color='#e74c3c')

ax.set_ylabel('Score')
ax.set_title('R2 vs Adjusted R2: Impact of Useless Features', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0.7, 0.95)
ax.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Key Observations from the Comparison

### 4.1 R² vs. Adjusted R² Behavior
- **$R^2$ Score:** Stayed almost the same or **slightly increased** for Model 2. This is because standard $R^2$ always rewards models for having more variables, even if those variables are random numbers.
- **Adjusted $R^2$ Score:** **Decreased significantly** for Model 2. It successfully penalized the model for adding features that did not provide actual predictive value.

### 4.2 Error Metrics (MAE, MSE, RMSE) Behavior
- Because the random noise features allowed the model to overfit slightly, the error metrics on the test set showed a slight deterioration (errors increased) for Model 2 compared to Model 1.

---

## 5. Summary Reference Guide

Use this guide to select the right metric for your regression tasks:

| Metric | When to Use | Key Trait |
| :--- | :--- | :--- |
| **MAE** | When you want an easy-to-understand error in original units, and you do **not** want outliers to dominate the results. | Robust to outliers |
| **MSE** | Mainly internally as a **loss function** for mathematical optimization (gradient descent). | Differentiable |
| **RMSE** | When you want to evaluate overall model performance in original units, but still want to **penalize large errors**. | Standard deviation of residuals |
| **R² Score** | When you need to understand the **percentage of variance** explained by the model compared to a simple baseline. | Relative comparison |
| **Adjusted R²** | When evaluating **multiple regression models** with varying numbers of features to prevent artificial score inflation. | Penalizes feature complexity |

---

> ⚠️ **Warning:** Always use **Adjusted R²** instead of standard **R²** when performing feature selection or comparing models with different numbers of predictors!